In [7]:
import json
import requests
from src.data.make_dataset import build_interim_dataset, load_params
from src.features.build_features import build_features_dataset
from src.config import resolve_path
import pandas as pd

# Generando datos nuevos

In [3]:
newData = resolve_path('data/external')
datasetPath = resolve_path('data/external/interim')
featuresPath = resolve_path('data/external/processed')
paramsPath = resolve_path('params.yaml')
params = load_params(paramsPath)
build_interim_dataset(params,raw_dir=newData, out_dir=datasetPath)


Validación completada: dataset consistente.


PosixPath('/home/angelbri/MLopsProject/MLops_Equipo12/data/external/interim/mibici_data.parquet')

In [5]:
datasetPath = resolve_path('data/external/interim/mibici_data.parquet')
featuresPath = resolve_path('data/external/processed')
build_features_dataset(params,datasetPath,featuresPath)

/home/angelbri/MLopsProject/MLops_Equipo12/data/external/interim/mibici_data.parquet


PosixPath('/home/angelbri/MLopsProject/MLops_Equipo12/data/external/processed/mibici_features.parquet')

In [8]:
proccessedData = resolve_path('data/external/processed/mibici_features.parquet')
df = pd.read_parquet(proccessedData)
df.head()

,Origen_Id,fecha_hora,demanda,lag_1h,lag_2h,lag_6h,lag_12h,lag_24h,lag_7d,media_24h,...,hora,dia_semana,mes,es_fin_semana,hour_sin,hour_cos,dow_sin,dow_cos,mes_sin,mes_cos
0,10,2026-10-08 00:00:00,0,1.0,0.0,4.0,1.0,2.0,0.0,2.583333,...,0,3,10,0,0.000000,1.000000,0.433884,-0.900969,-0.866025,0.5
1,10,2026-10-08 01:00:00,0,0.0,1.0,3.0,4.0,0.0,0.0,2.500000,...,1,3,10,0,0.258819,0.965926,0.433884,-0.900969,-0.866025,0.5
2,10,2026-10-08 02:00:00,0,0.0,0.0,6.0,4.0,0.0,0.0,2.500000,...,2,3,10,0,0.500000,0.866025,0.433884,-0.900969,-0.866025,0.5
3,10,2026-10-08 03:00:00,0,0.0,0.0,1.0,2.0,0.0,0.0,2.500000,...,3,3,10,0,0.707107,0.707107,0.433884,-0.900969,-0.866025,0.5
4,10,2026-10-08 04:00:00,0,0.0,0.0,0.0,6.0,0.0,0.0,2.500000,...,4,3,10,0,0.866025,0.500000,0.433884,-0.900969,-0.866025,0.5


# Probando API

In [16]:
features = [
    "lag_1h",
    "lag_2h",
    "lag_6h",
    "lag_12h",
    "lag_24h",
    "lag_7d",
    "media_24h",
    "rolling_max_6h",
    "rolling_max_12h",
    "rolling_max_24h",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "mes_sin",
    "mes_cos",
    "Origen_Id",
    "es_fin_semana",
]

payload = df.loc[2, features].to_dict()

payload

{'lag_1h': 0.0,
 'lag_2h': 0.0,
 'lag_6h': 6.0,
 'lag_12h': 4.0,
 'lag_24h': 0.0,
 'lag_7d': 0.0,
 'media_24h': 2.5,
 'rolling_max_6h': 6.0,
 'rolling_max_12h': 6.0,
 'rolling_max_24h': 10.0,
 'hour_sin': 0.49999999999999994,
 'hour_cos': 0.8660254037844387,
 'dow_sin': 0.43388373911755823,
 'dow_cos': -0.900968867902419,
 'mes_sin': -0.8660254037844386,
 'mes_cos': 0.5000000000000001,
 'Origen_Id': '10',
 'es_fin_semana': 0}

In [19]:
r = requests.post(
    "http://localhost:8000/predict",
    json=payload,
)

print(r.status_code)
print(r.json())

200
{'predictions': [-0.028301460668444633]}
